In [42]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [43]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Generate Test Dataset

In [44]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [45]:
dataset =  generate_dataset()

dataset

[{'task': "Write a Python function that parses an AWS CloudWatch log entry and extracts the timestamp, log level, and message. The log format is '[TIMESTAMP] [LEVEL] MESSAGE'."},
 {'task': "Create a JSON object that represents an AWS S3 bucket policy allowing read-only access to objects with a specific prefix 'public/' for any principal."},
 {'task': "Write a regular expression that matches valid AWS IAM role ARNs in the format 'arn:aws:iam::123456789012:role/RoleName'."}]

In [46]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## Graders

In [47]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## Evaluate Prompt

In [48]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [49]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [50]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [51]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score: 6.333333333333333
[
  {
    "output": "# AWS CloudWatch Log Parser\n\nHere's a comprehensive solution for parsing AWS CloudWatch log entries:\n\n```python\nimport re\nfrom datetime import datetime\nfrom typing import Dict, Optional, Tuple\nfrom enum import Enum\n\n\nclass LogLevel(Enum):\n    \"\"\"Enum for valid log levels\"\"\"\n    DEBUG = \"DEBUG\"\n    INFO = \"INFO\"\n    WARNING = \"WARNING\"\n    ERROR = \"ERROR\"\n    CRITICAL = \"CRITICAL\"\n\n\ndef parse_cloudwatch_log(log_entry: str) -> Optional[Dict[str, any]]:\n    \"\"\"\n    Parse an AWS CloudWatch log entry and extract timestamp, log level, and message.\n    \n    Args:\n        log_entry: A log entry string in format '[TIMESTAMP] [LEVEL] MESSAGE'\n        \n    Returns:\n        A dictionary with keys 'timestamp', 'level', and 'message', or None if parsing fails\n        \n    Example:\n        >>> log = '[2024-01-15 10:30:45.123] [ERROR] Database connection failed'\n        >>> parse_cloudwatch_log(log

In [52]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS CloudWatch Log Parser\n\nHere's a comprehensive solution for parsing AWS CloudWatch log entries:\n\n```python\nimport re\nfrom datetime import datetime\nfrom typing import Dict, Optional, Tuple\nfrom enum import Enum\n\n\nclass LogLevel(Enum):\n    \"\"\"Enum for valid log levels\"\"\"\n    DEBUG = \"DEBUG\"\n    INFO = \"INFO\"\n    WARNING = \"WARNING\"\n    ERROR = \"ERROR\"\n    CRITICAL = \"CRITICAL\"\n\n\ndef parse_cloudwatch_log(log_entry: str) -> Optional[Dict[str, any]]:\n    \"\"\"\n    Parse an AWS CloudWatch log entry and extract timestamp, log level, and message.\n    \n    Args:\n        log_entry: A log entry string in format '[TIMESTAMP] [LEVEL] MESSAGE'\n        \n    Returns:\n        A dictionary with keys 'timestamp', 'level', and 'message', or None if parsing fails\n        \n    Example:\n        >>> log = '[2024-01-15 10:30:45.123] [ERROR] Database connection failed'\n        >>> parse_cloudwatch_log(log)\n        {\n            'timest